# Step 10 & 11: Performance Monitoring & Drift Detection
## Operationalizing ML: When is the Model Degrading?

In production, models degrade due to two fundamentally different phenomena:
1. **Data Drift**: $P(X)$ changes. Input feature distributions shift (e.g., changes in rider request volume, weather, special events).
2. **Concept Drift / Performance Degradation**: $P(Y|X)$ changes. The underlying relationship between input features and demand shifts, causing model accuracy to degrade.

> **CRITICAL RULE**: Detecting data drift does **NOT** automatically mean concept drift has occurred. A model may be robust to data drift. Therefore, we monitor both feature stability (PSI, KS-test) and actual operational performance (Rolling MAE, RMSE, and Bias).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from src.drift import calculate_psi, calculate_ks_test, compute_rolling_metrics, detect_sustained_degradation
from src.features import build_feature_pipeline
from src.data_loader import load_processed_demand, split_chronological
from src.config import STREAM_PREDICTIONS_PARQUET, DRIFT_PSI_THRESHOLD

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig_size = (12, 5)

print("Loading stream prediction audit log...")
pred_df = pd.read_parquet(STREAM_PREDICTIONS_PARQUET)
print(f"Loaded {len(pred_df):,} audit records.")
pred_df.head()


### 1. Rolling Performance Tracking (MAE, RMSE, Bias)
We calculate 24-hour rolling metrics (window of 96 intervals) across the streaming timeline.


In [ ]:
rolling_summary = compute_rolling_metrics(pred_df, window_size=96)
rolling_summary.head(10)


### 2. Visualize Rolling MAE & Degradation Thresholds
We set an alert control limit at 15% above the baseline validation MAE.


In [ ]:
baseline_val_mae = rolling_summary['mean_abs_error'].median()
control_limit = baseline_val_mae * 1.15

plt.figure(figsize=fig_size)
plt.plot(rolling_summary['prediction_timestamp'], rolling_summary['rolling_mae'], 
         color='#1f77b4', lw=2, label='Rolling 24h MAE')
plt.axhline(baseline_val_mae, color='green', linestyle='--', label=f'Baseline MAE ({baseline_val_mae:.2f})')
plt.axhline(control_limit, color='red', linestyle='--', label=f'Alert Limit (+15%: {control_limit:.2f})')

plt.title("Rolling 24-Hour MAE Over Simulated Stream Timeline", fontsize=14, fontweight='bold')
plt.xlabel("Timestamp", fontsize=12)
plt.ylabel("Mean Absolute Error (MAE)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 3. Rolling Bias (Mean Error): Monitoring Systematic Over/Under Prediction
Bias tracks whether the model consistently over-predicts (positive) or under-predicts (negative).


In [ ]:
plt.figure(figsize=fig_size)
plt.plot(rolling_summary['prediction_timestamp'], rolling_summary['rolling_bias'], 
         color='#ff7f0e', lw=2, label='Rolling 24h Bias (Predicted - Actual)')
plt.axhline(0, color='black', linestyle=':', lw=1.5)
plt.title("Rolling Prediction Bias (Mean Error) Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Timestamp", fontsize=12)
plt.ylabel("Bias (Rides)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 4. Input Data Drift Detection (PSI & KS-Test)
We compute Population Stability Index (PSI) and Kolmogorov-Smirnov test between the Training reference distribution and the Streaming test window.


In [ ]:
grid_df = load_processed_demand()
feat_df = build_feature_pipeline(grid_df, drop_burn_in=True)
train_df, _, test_df = split_chronological(feat_df)

features_to_monitor = ['lag_1', 'lag_4', 'lag_96', 'rolling_mean_4']
drift_results = []

for feat in features_to_monitor:
    ref_vals = train_df[feat].to_numpy()
    cur_vals = test_df[feat].to_numpy()
    
    psi_val = calculate_psi(ref_vals, cur_vals)
    ks_res = calculate_ks_test(ref_vals, cur_vals)
    
    drift_status = "Significant Drift" if psi_val >= 0.25 else ("Moderate Shift" if psi_val >= 0.10 else "Stable")
    
    drift_results.append({
        'Feature': feat,
        'PSI': psi_val,
        'PSI_Status': drift_status,
        'KS_Statistic': ks_res['ks_statistic'],
        'KS_P_Value': ks_res['p_value'],
        'KS_Drift': ks_res['is_drift']
    })

drift_df = pd.DataFrame(drift_results)
print("Data Drift Profiling Summary:")
print(drift_df.to_string(index=False))


### 5. Sustained Performance Degradation Check
We check whether rolling MAE sustained an increase above the control limit for consecutive periods.


In [ ]:
has_degraded, degraded_idx = detect_sustained_degradation(
    rolling_summary['rolling_mae'],
    baseline_mae=baseline_val_mae,
    threshold_pct=0.15,
    consecutive_periods=4
)

print(f"Sustained Performance Degradation Detected: {has_degraded}")
if has_degraded:
    print(f"Triggering Continual Learning / Model Update Workflow at timestamp: "
          f"{rolling_summary.loc[degraded_idx[0], 'prediction_timestamp']}")
else:
    print("Model performance remains within acceptable operational boundaries.")
